# Supplementary: Low-rank approximation via SVD

Companion to `../04_outer_products_low_rank.md`.

**Goal**: build hands-on intuition for *what* 'low-rank approximation' means by truncating the SVD and watching the reconstruction error fall as you keep more singular values. This is the foundation behind LoRA, embedding compression, and SVD-based weight pruning.

**Theory recap**: for any real matrix `A`, the SVD `A = U Σ V^T` lets you write

$$A = \sum_{i=1}^{\min(m,n)} \sigma_i\, u_i v_i^T$$

i.e. `A` is a sum of rank-1 outer products weighted by singular values. The **Eckart–Young theorem** says the best rank-`k` approximation of `A` (in Frobenius or spectral norm) is obtained by keeping the top `k` terms of this sum and zeroing the rest. Truncating SVD is *optimal*.

In [ ]:
import torch
import matplotlib.pyplot as plt

torch.manual_seed(42)

## Part 1 — A random matrix

First we'll do the textbook exercise: take a random 100×100 Gaussian matrix and study its low-rank approximations.

In [ ]:
n = 100
A = torch.randn(n, n)
print(f'A.shape: {A.shape}')
print(f'‖A‖_F:   {torch.linalg.norm(A, ord="fro"):.2f}')

### Compute the SVD

PyTorch returns `U`, `S`, `Vh` where `Vh = V^T`. So `A = U @ diag(S) @ Vh`.

In [ ]:
U, S, Vh = torch.linalg.svd(A, full_matrices=False)
print(f'U:  {U.shape}')   # (n, min(n,n)) = (100, 100)
print(f'S:  {S.shape}')   # (min(n,n),)   = (100,)  — singular values, sorted descending
print(f'Vh: {Vh.shape}')  # (min(n,n), n) = (100, 100)

# Verify
A_reconstructed = U @ torch.diag(S) @ Vh
print(f'\nReconstruction error (full SVD): {torch.linalg.norm(A - A_reconstructed):.2e}')

**Expected**: shapes `(100, 100)`, `(100,)`, `(100, 100)`. Reconstruction error should be ~`1e-5` or smaller — round-trip through SVD recovers the matrix to floating-point precision.

### Truncate to rank `k`

The rank-`k` reconstruction is `U[:, :k] @ diag(S[:k]) @ Vh[:k, :]`. Equivalently, `Σ_{i=1}^k σ_i u_i v_i^T`.

In [ ]:
def low_rank_approx(U, S, Vh, k):
    return U[:, :k] @ torch.diag(S[:k]) @ Vh[:k, :]

for k in [1, 5, 20, 100]:
    A_k = low_rank_approx(U, S, Vh, k)
    err = torch.linalg.norm(A - A_k)
    rel_err = err / torch.linalg.norm(A)
    print(f'k = {k:3d}   Frobenius error = {err:6.2f}   relative = {rel_err:.3f}')

**Expected pattern**: error decreases as `k` grows but only slowly for a *random* matrix, because random Gaussian matrices have a roughly flat singular-value spectrum (no direction is special). At `k=100` the error hits ~0 (full reconstruction).

If random matrices reconstruct slowly, why does low-rank approximation help in practice? Because real DL weight matrices are *not* random — they have heavily skewed singular value spectra after training, with most of the 'mass' in the top few singular values. We'll demonstrate this next.

### Sweep `k` and plot Frobenius error

In [ ]:
ks = list(range(1, n + 1))
errs_random = [torch.linalg.norm(A - low_rank_approx(U, S, Vh, k)).item() for k in ks]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(ks, errs_random)
ax1.set_xlabel('rank k')
ax1.set_ylabel('‖A - A_k‖_F (Frobenius error)')
ax1.set_title('Random Gaussian matrix: error vs k')
ax1.grid(True)

ax2.semilogy(range(1, n + 1), S.numpy())
ax2.set_xlabel('singular value index i')
ax2.set_ylabel('σ_i (log scale)')
ax2.set_title('Random Gaussian matrix: singular value spectrum')
ax2.grid(True)
plt.tight_layout()
plt.show()

**What you should see**: a roughly linear (or slow-decaying) error curve, and a singular-value spectrum that's nearly flat in log scale (Marchenko-Pastur distribution for random Gaussian matrices). No clean rank cutoff — every singular value carries meaningful information.

## Part 2 — A 'naturally low-rank' matrix

Now construct a matrix that *is* approximately low-rank: sum of a few outer products plus a small noise floor. This mimics what DL weight matrices actually look like.

In [ ]:
# 5 dominant directions + small isotropic noise
true_rank = 5
B = torch.randn(n, true_rank) @ torch.randn(true_rank, n)
B = B + 0.1 * torch.randn(n, n)

U_b, S_b, Vh_b = torch.linalg.svd(B, full_matrices=False)

for k in [1, 5, 20, 100]:
    B_k = low_rank_approx(U_b, S_b, Vh_b, k)
    err = torch.linalg.norm(B - B_k)
    rel_err = err / torch.linalg.norm(B)
    print(f'k = {k:3d}   Frobenius error = {err:6.2f}   relative = {rel_err:.3f}')

**Expected**: error drops *sharply* between `k=1` and `k=5` (capturing the dominant directions) and then plateaus — the additional rank only chases the small noise floor. This is the signature of a near-low-rank matrix.

In [ ]:
errs_lowrank = [torch.linalg.norm(B - low_rank_approx(U_b, S_b, Vh_b, k)).item() for k in ks]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(ks, errs_random, label='random Gaussian')
ax1.plot(ks, errs_lowrank, label='near-low-rank')
ax1.set_xlabel('rank k')
ax1.set_ylabel('‖A - A_k‖_F')
ax1.set_title('Frobenius error vs k')
ax1.legend()
ax1.grid(True)

ax2.semilogy(range(1, n + 1), S.numpy(), label='random Gaussian')
ax2.semilogy(range(1, n + 1), S_b.numpy(), label='near-low-rank')
ax2.set_xlabel('singular value index i')
ax2.set_ylabel('σ_i (log scale)')
ax2.set_title('Singular value spectra')
ax2.legend()
ax2.grid(True)
plt.tight_layout()
plt.show()

**What you should see**:
- The near-low-rank error curve drops fast and flattens around the noise floor.
- The near-low-rank singular value spectrum has a sharp 'knee' at index 5: the first 5 σ_i are large, the rest are small. The knee tells you the effective rank.

When you see a paper plot 'singular value spectrum of W' and notice the sharp drop, *that's the data point* showing the matrix is approximately low-rank — and therefore compressible / amenable to LoRA-style updates.

## Part 3 — Tying back to LoRA

LoRA's hypothesis: when fine-tuning a pretrained weight `W ∈ R^(d×d)` with an update `ΔW`, the *update* is approximately low-rank (typical effective rank `r = 8`–`64`). The base `W` itself isn't necessarily low-rank, but the *change* during fine-tuning lives in a small subspace of directions.

Why this is plausible: fine-tuning adapts the model to a narrow distribution shift. Most of the pretrained weight structure is already correct; only a few directions need adjustment. SVD on real LoRA `ΔW = BA` matrices does show this behavior empirically — first few σ_i large, rest negligible.

Reading exercise: open the LoRA paper (Hu et al. 2021), look at Figure 7 (the singular-value spectrum of fine-tuning updates). It looks exactly like the 'near-low-rank' curve above.

## Self-check

1. For a 1000×1000 random Gaussian matrix, what's the rough *relative* error at `k=10`? (Predict, then run.)
2. If a matrix has singular values `[10, 9.5, 9, 0.01, 0.01, 0.01]`, what is the relative Frobenius error at `k=3`?
3. Why is the optimal rank-`k` approximation an SVD truncation rather than, say, the rank-`k` matrix that minimizes operator norm error? (Hint: it's both — the Eckart-Young theorem covers Frobenius and spectral norm.)
4. The full rank-100 'reconstruction' of a 100×100 matrix uses 100·100 + 100 + 100·100 = 20100 numbers (U, S, Vh). The original A has only 10000 numbers. Why is this still useful?

## Answers

1. ~96–98%. The error decays roughly like `√(1 - k/n)` for random Gaussian, so at k/n=0.01 you'd retain ~99% of the norm — meaning ~99% of the error remains. SVD truncation barely helps for random matrices; this is the expected behavior.
2. `‖A - A_3‖_F = √(0.01² + 0.01² + 0.01²) ≈ 0.017`. `‖A‖_F = √(10² + 9.5² + 9² + 3·0.01²) ≈ 16.5`. Relative ≈ 0.001 (0.1%). Sharp low-rank structure → tiny error from truncation.
3. Eckart-Young proves SVD truncation is optimal for *both* the Frobenius norm and the spectral norm (operator 2-norm). So you don't have to choose — the same truncation is best in both senses. (For other matrix norms, the optimal low-rank approximation may differ.)
4. The 'storage win' of SVD isn't from the full decomposition — it's from *truncation*. A rank-`k` approximation needs `(m + n) · k + k` numbers (the trimmed `U`, `Vh`, and `S`). For `m = n = 100, k = 5`: `5·100 + 5·100 + 5 = 1005` numbers vs. 10000 — 10× compression with small error loss. Same as why LoRA is `(d + d) · r` parameters per layer instead of `d²`.